# Student Engagement Detection using Swin Transformer

This notebook implements a student engagement detection system using the Swin Transformer architecture. We map 6 sub-categories into 3 main classes (Highly Engaged, Moderately Engaged, Disengaged) to compare results with the original research paper.

In [ ]:
# Step 1: Install and Import Libraries
!pip install -q transformers torchvision scikit-learn matplotlib seaborn tqdm

import os
import shutil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import SwinForImageClassification, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm

In [ ]:
# Step 2: Restructure Dataset
source_dir = 'Student-engagement-dataset'
target_dir = 'Dataset_3_Classes'

# Define the mapping
mapping = {
    'Highly_Engaged': [
        'Engaged/Focused'
    ],
    'Moderately_Engaged': [
        'Engaged/Confused',
        'Engaged/Frustrated'
    ],
    'Disengaged': [
        'Not Engaged/Bored',
        'Not Engaged/Drowsy',
        'Not Engaged/Looking Away'
    ]
}

def restructure_dataset(src, dst, mapping):
    if not os.path.exists(dst):
        os.makedirs(dst)
    
    for class_name, subfolders in mapping.items():
        class_path = os.path.join(dst, class_name)
        if not os.path.exists(class_path):
            os.makedirs(class_path)
        
        for sub in subfolders:
            sub_src_path = os.path.join(src, sub)
            if os.path.exists(sub_src_path):
                print(f"Copying images from {sub} to {class_name}...")
                for img in os.listdir(sub_src_path):
                    if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                        shutil.copy(os.path.join(sub_src_path, img), os.path.join(class_path, img))
            else:
                print(f"Warning: {sub_src_path} not found.")

restructure_dataset(source_dir, target_dir, mapping)

In [ ]:
# Step 3: Data Loading & Augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=target_dir, transform=transform)
print(f"Total images: {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}")

# Split into 70% Train, 15% Val, 15% Test
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# Step 4: Load Swin Transformer
model_name = "microsoft/swin-tiny-patch4-window7-224"
model = SwinForImageClassification.from_pretrained(
    model_name,
    num_labels=3,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

In [ ]:
# Step 5: Training Loop
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
num_epochs = 15
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Step 6: Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        preds = torch.argmax(outputs.logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average='macro')

print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")
print(f"Final Macro F1-Score: {macro_f1 * 100:.2f}%")

In [ ]:
# Step 7: Visualization
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=full_dataset.classes, 
            yticklabels=full_dataset.classes)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()